<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/LongTermShortTermMemory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.messages import BaseMessage
from langchain_core.tools import create_retriever_tool
from langchain_community.document_loaders import TextLoader
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware.types import after_model, before_model
from langchain.agents import AgentState
from langgraph.runtime import Runtime
from google.colab import userdata
from pydantic import SecretStr
from typing import List, TypedDict
from IPython.display import Image
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
import json

In [4]:

def print_conversation(messages: List[BaseMessage]):
  for message in messages:
    message.pretty_print()

In [5]:
model = ChatOpenAI(model="gpt-5-nano", api_key=userdata.get('OPENAI_KEY'))

In [6]:
checkpointer = InMemorySaver()
store = InMemoryStore()

In [11]:
class TravelConsultantContext(TypedDict):
  user_id: str


@before_model
def before_model_func(state: AgentState, runtime: TravelConsultantContext):
  print(f"BEFORE MODEL FUNC.  USER ID ---> {runtime.context["user_id"]}")

@after_model
def after_model_func(state: AgentState, runtime: Runtime):
  print("AFTER MODEL FUNC")


@tool
def save_preference(key: str, preference: str, runtime: ToolRuntime[TravelConsultantContext]) -> str:
  """
  Save the preferances about the user like dietary, seating, alergies, etc.
  """
  runtime.store.put(
      namespace=("users", runtime.context["user_id"], "preferences"),
      key=key,
      value={"preference": preference}
  )
  return "OK"


In [23]:
DESTINATION_TO_TRAVEL = {"Paris": {"season": "spring", "price_range": {"min": 100, "max": 200, "currency": "EUR"}}, "Rome": {"season": "summer", "price_range": {"min": 80, "max": 150, "currency": "EUR"}}, "Barcelona": {"season": "spring", "price_range": {"min": 90, "max": 180, "currency": "EUR"}}, "Vienna": {"season": "autumn", "price_range": {"min": 90, "max": 170, "currency": "EUR"}}}

CATALOGUE = {
    "destinations": DESTINATION_TO_TRAVEL,
    "services": [
        "Private-jet transfers via NetJets and VistaJet",
        "Forbes 5-Star and Relais & Chateaux properties only",
        "Personal butler and dedicated 24/7 destination manager",
        "Michelin-experience curation (typically 2-3* venues)",
    ]
}

CATALOGUE_AS_CONTEXT = json.dumps(CATALOGUE, indent=2)

SYSTEM_PROMPT = f"""You are **Jacque**, the AI concierge of an ultra-luxury travel atelier.
You speak with warm restraint - think a Parisian maître d'hotel, never a salesperson.

# Catalogue you may sell from
{CATALOGUE_AS_CONTEXT}

# Operating rules
1. When the guest reveals a durable preference that was not previously recalled (allergy, favourite activity, etc.), call `{save_preference.name}`. Use stable, lowercase keys (e.g. `dietary`, `seat_preference`, `destination_to_travel`).

# Topical guardrails - politely refuse and steer back
- Budget travel, hostels, backpacking, cheap flights -> "Our atelier is positioned exclusively in the ultra-luxury segment; may I suggest one of our signature retreats instead?"
- Competing agencies (Abercrombie & Kent, Black Tomato, etc.) -> decline to compare; redirect.
- Politics, religion, controversial public figures -> "I keep my counsel to the art of travel."
- Medical, legal or financial advice -> recommend a qualified professional.
"""

In [24]:
agent = create_agent(
    model=model,
    checkpointer=checkpointer,
    store=store,
    context_schema=TravelConsultantContext,
    tools=[save_preference],
    middleware=[before_model_func, after_model_func],
    system_prompt=SYSTEM_PROMPT,
    debug=True
)

In [ ]:
response_1 = agent.invoke(
    input={
      "messages": [
          HumanMessage("Hello, my name is Ivan Uzunov.")
      ]
    },
    config={
      "configurable": {
          "thread_id": "thread_id_3"
      }
    },
    context={
      "user_id": "user_id_1"
    }
  )

In [ ]:
response_2 = agent.invoke(
    input={
      "messages": [
          HumanMessage(content="I want to travel in austria. What will you offer me?")
      ]
  },
  config={
      "configurable": {
          "thread_id": "thread_id_3"
      }
    },
    context={
      "user_id": "user_id_1"
    }
)

In [ ]:
display(Image(agent.get_graph().draw_mermaid_png()))


In [ ]:
for x in checkpointer.list({"configurable": {"thread_id": "thread_id_1"}}):
  print(x)